In [1]:
!pip -q install transformers sentencepiece sacremoses sacrebleu

In [2]:
import torch
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [3]:
def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSeq2SeqLM.from_pretrained(name).to(device).eval()
    return tok, model

def translate(texts, tok, model, prefix="", beams=4):
    batch = tok([prefix + t for t in texts], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**batch, num_beams=beams, max_new_tokens=128)
    return tok.batch_decode(out, skip_special_tokens=True)

sentences = [
    "The weather is very nice today.",
    "I am learning natural language processing at college.",
    "Machine translation helps people communicate across languages.",
]

In [4]:
# ---------- MarianMT: English -> Hindi ----------
tok_hi, m_hi = load("Helsinki-NLP/opus-mt-en-hi")
hi = translate(sentences, tok_hi, m_hi)
for s, t in zip(sentences, hi): print(f"EN: {s}\nHI: {t}\n")

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  306MB            

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EN: The weather is very nice today.
HI: मौसम आज बहुत अच्छा है.

EN: I am learning natural language processing at college.
HI: मैं कॉलेज में स्वाभाविक भाषा प्रक्रिया सीख रहा हूँ.

EN: Machine translation helps people communicate across languages.
HI: मशीन अनुवाद से लोगों को भाषा में बात करने में मदद मिलती है ।



In [5]:
# ---------- MarianMT: English -> French ----------
tok_fr, m_fr = load("Helsinki-NLP/opus-mt-en-fr")
fr = translate(sentences, tok_fr, m_fr)
for s, t in zip(sentences, fr): print(f"EN: {s}\nFR: {t}\n")

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  301MB            

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EN: The weather is very nice today.
FR: Le temps est très beau aujourd'hui.

EN: I am learning natural language processing at college.
FR: J'apprends le traitement naturel des langues à l'université.

EN: Machine translation helps people communicate across languages.
FR: La traduction automatique aide les gens à communiquer entre les langues.



In [6]:
# ---------- T5-small: English -> German / French (task prefix) ----------
tok_t5, m_t5 = load("t5-small")
de_t5 = translate(sentences, tok_t5, m_t5, prefix="translate English to German: ")
fr_t5 = translate(sentences, tok_t5, m_t5, prefix="translate English to French: ")
for s, d, f in zip(sentences, de_t5, fr_t5):
    print(f"EN: {s}\nDE (T5): {d}\nFR (T5): {f}\n")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

EN: The weather is very nice today.
DE (T5): Das Wetter ist heute sehr schön.
FR (T5): Le temps est très agréable aujourd'hui.

EN: I am learning natural language processing at college.
DE (T5): Ich lerne natürliche Sprachenverarbeitung am Hochschulabschluss.
FR (T5): J'ai appris le traitement de la langue naturelle au collège.

EN: Machine translation helps people communicate across languages.
DE (T5): Automatische Übersetzung hilft Menschen, über Sprachen hinweg zu kommunizieren.
FR (T5): La traduction automatique aide les gens à communiquer entre les langues.



In [7]:
# ---------- Evaluation with BLEU (English -> French reference translations) ----------
refs_fr = [
    "Il fait très beau aujourd'hui.",
    "J'apprends le traitement automatique du langage naturel à l'université.",
    "La traduction automatique aide les gens à communiquer entre les langues.",
]
bleu_marian = sacrebleu.corpus_bleu(fr, [refs_fr]).score
bleu_t5 = sacrebleu.corpus_bleu(fr_t5, [refs_fr]).score
display(pd.DataFrame({"model": ["MarianMT (opus-mt-en-fr)", "T5-small"], "BLEU": [round(bleu_marian, 2), round(bleu_t5, 2)]}))

,model,BLEU
0,MarianMT (opus-mt-en-fr),66.71
1,T5-small,47.66


In [8]:
# Translate your own sentence
my_text = ["Students should practice coding every day."]
print("Hindi :", translate(my_text, tok_hi, m_hi)[0])
print("French:", translate(my_text, tok_fr, m_fr)[0])

[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hindi : विद्यार्थियों को हर दिन काम करना चाहिए ।
French: Les élèves devraient pratiquer le codage tous les jours.
